In [ ]:
# ============================================================================
# Zebrafish & Lamprey Cerebellum Spatial Transcriptomics Analysis
# Simplified version - Core processing steps
# ============================================================================

# Load required libraries
library(Seurat)      # Single-cell analysis framework
library(dplyr)       # Data manipulation
library(ggplot2)     # Plotting

# ============================================================================
# Section 1: Load Zebrafish Spatial Data
# ============================================================================
zfsp <- readRDS("/data/users/niuzhiwei1/online/00物种数据/11斑马鱼空间芯片/04周涛处理/05adata.region.0107_基因大写_同源转换.rds")

# Check sample distribution across slices
table(zfsp$slice_code)

# Set region as identity for downstream analysis
Idents(zfsp) <- "region"

# ============================================================================
# Section 2: Spatial DimPlot for Each Slice
# Visualize spatial distribution of cell clusters
# ============================================================================
slice_codes <- c("D03558D2.left", "D03558D2.right", "D03558D4.left", "D03558D4.right")

for (slice in slice_codes) {
    # Subset data for each slice and create spatial DimPlot
    p <- DimPlot(subset(zfsp, subset = slice_code == slice), 
                reduction = "spatial", 
                label = TRUE)
    
    # Save plot to PDF
    pdf_filename <- paste0("zebrafish.", slice, ".bin.region.pdf")
    pdf(pdf_filename, width = 11, height = 9)
    print(p)
    dev.off()
}

# ============================================================================
# Section 3: Load Combined Zebrafish-Lamprey Cerebellum Data
# ============================================================================
cball <- readRDS("01斑马鱼空间_七鳃鳗空间_基因交集_cb_分析.rds")

# Check region distribution in combined dataset
table(cball@meta.data$region)

# ============================================================================
# Section 4: UMAP Visualization
# ============================================================================
# Overall UMAP colored by cluster
pdf("zf.les.cb.all.umap.pdf", width = 5, height = 5)
DimPlot(cball, label = TRUE) + NoLegend()
dev.off()

# Split UMAP by species (zebrafish vs lamprey)
pdf("zf.les.cb.split.umap.pdf", width = 10, height = 5)
DimPlot(cball, label = TRUE, split.by = "species") + NoLegend()
dev.off()

# Split UMAP by region
pdf("zf.les.cb.split.region.label.umap.pdf", width = 12, height = 5)
DimPlot(cball, split.by = "species")
dev.off()

# ============================================================================
# Section 5: Highlight Specific Clusters on Spatial Plots
# Highlight zebrafish clusters 0, 1, 8 (purkinje-like cells)
# ============================================================================
Idents(cball) <- "region"

# Identify cells in target clusters from zebrafish
c1 <- rownames(cball@meta.data[
    which(cball@meta.data$species == "zfish_spa" & 
          cball@meta.data$seurat_clusters %in% c(0, 1, 8)),
])

# Create spatial highlight plots for each slice
for (slice in slice_codes) {
    p <- DimPlot(subset(zfsp, subset = slice_code == slice),
                reduction = "spatial",
                cells.highlight = c1)
    
    pdf_filename <- paste0("zebrafish.", slice, ".bin.c018.pdf")
    pdf(pdf_filename, width = 6, height = 9)
    print(p)
    dev.off()
}
